# Lab 4 — Deploy the fine-tuned OpenVLA to a real-time endpoint

The endpoint runs the PyTorch inference DLC with the custom handler stored in the model package
(`hf_merged/code/inference.py`). OpenVLA needs `trust_remote_code` and action-token decoding, so the
serverless lab's LMI/vLLM container doesn't apply.

* request: `{"image": <base64 JPEG/PNG>, "instruction": "..."}`
* response: `{"action": [7 floats in [-1, 1]], "action_unnormalized": [...]}`

The endpoint config lists several GPU instance types as `InstancePools` (`config.ENDPOINT_INSTANCE_TYPES`):
SageMaker provisions the first type with capacity. Startup includes downloading 15 GB of weights.

In [ ]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

import config

sess = Session()
try:
    role = get_execution_role()
except ValueError:
    role = boto3.client("iam").get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

region = sess.boto_region_name
bucket_name = sess.default_bucket()
s3_root = config.s3_root(bucket_name, sess.default_bucket_prefix)
dataset_s3_uri = f"{s3_root}/datasets/{config.DATASET_PREFIX}/"
s3 = boto3.client("s3", region_name=region)
sm_client = boto3.client("sagemaker", region_name=region)

print(f"role:     {role}")
print(f"region:   {region}")
print(f"run mode: {config.RUN_MODE} -> {config.TRAIN['instance_type']}")
print(f"s3 root:  {s3_root}")

In [ ]:
def latest_package(status=None):
    kwargs = {"ModelPackageGroupName": config.MODEL_PACKAGE_GROUP, "SortBy": "CreationTime", "SortOrder": "Descending"}
    if status:
        kwargs["ModelApprovalStatus"] = status
    pkgs = sm_client.list_model_packages(**kwargs)["ModelPackageSummaryList"]
    if not pkgs:
        raise RuntimeError(f"no {status or ''} model package in {config.MODEL_PACKAGE_GROUP}; run the previous notebook")
    return sm_client.describe_model_package(ModelPackageName=pkgs[0]["ModelPackageArn"])

In [ ]:
# Only Approved packages are deployed. Lab 3 approves a package that beats the base model
# (smoke) or the base model and the label prior (full). FORCE_APPROVE = True approves the latest package here
# instead, e.g. to test deployment with a package lab 3 rejected; the override is recorded in its description.
FORCE_APPROVE = False

import time

package = latest_package()
if package["ModelApprovalStatus"] != "Approved":
    if not FORCE_APPROVE:
        raise RuntimeError(f"latest package is {package['ModelApprovalStatus']} "
                           f"({package.get('ApprovalDescription', '')}); set FORCE_APPROVE = True to deploy it anyway")
    sm_client.update_model_package(
        ModelPackageArn=package["ModelPackageArn"], ModelApprovalStatus="Approved",
        ApprovalDescription=f"manual approval in 4-deployment (was {package['ModelApprovalStatus']}: "
                            f"{package.get('ApprovalDescription', '')})"[:1024])
    package = sm_client.describe_model_package(ModelPackageName=package["ModelPackageArn"])
name = f"{config.PROJECT}-{time.strftime('%Y%m%d-%H%M%S')}"
print(f"package: {package['ModelPackageArn']} ({package['ModelApprovalStatus']})\nendpoint: {name}")

In [ ]:
# Build the model from the approved package's container definition (same image, weights, handler, env), so the
# endpoint can use any type in config.ENDPOINT_INSTANCE_TYPES, not only those the package was registered with.
container = {k: v for k, v in package["InferenceSpecification"]["Containers"][0].items()
             if k in ("Image", "ModelDataSource", "Environment")}
sm_client.create_model(ModelName=name, ExecutionRoleArn=role, PrimaryContainer=container,
                       Tags=[{"Key": "ModelPackageArn", "Value": package["ModelPackageArn"]}])
sm_client.create_endpoint_config(
    EndpointConfigName=name,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": name,
        "InstancePools": [{"InstanceType": t, "Priority": i + 1} for i, t in enumerate(config.ENDPOINT_INSTANCE_TYPES)],
        "VariantInstanceProvisionTimeoutInSeconds": 3600,
        "InitialInstanceCount": 1,
        "ModelDataDownloadTimeoutInSeconds": 1800,
        "ContainerStartupHealthCheckTimeoutInSeconds": 1800,
    }],
)
sm_client.create_endpoint(EndpointName=name, EndpointConfigName=name)
print("creating endpoint (weights download + model load take a while)...")
try:
    sm_client.get_waiter("endpoint_in_service").wait(EndpointName=name, WaiterConfig={"Delay": 30, "MaxAttempts": 100})
except Exception:
    raise RuntimeError(sm_client.describe_endpoint(EndpointName=name).get("FailureReason", "endpoint failed"))
print(f"in service: {name}")

## Invoke with validation frames

In [ ]:
import base64
import io
import json

import numpy as np
from datasets import load_from_disk

runtime = boto3.client("sagemaker-runtime", region_name=region)
val = load_from_disk(f"data/{config.DATASET_PREFIX}")["validation"]


def predict(image, instruction):
    buf = io.BytesIO()
    image.save(buf, format="JPEG")
    body = json.dumps({"image": base64.b64encode(buf.getvalue()).decode(), "instruction": instruction})
    resp = runtime.invoke_endpoint(EndpointName=name, ContentType="application/json", Body=body)
    return json.loads(resp["Body"].read())


for row in val.select(range(5)):
    t = time.time()
    out = predict(row["image"], row["instruction"])
    print(f"pred x,y={np.round(out['action'][:2], 3)}  label x,y={np.round(row['action'][:2], 3)}  "
          f"({time.time() - t:.2f}s)")

Run **5-cleanup.ipynb** when you are done. The endpoint bills per hour until it is deleted.